# 🚀 Google Colab × Antigravity 智能联动实验室

本 Notebook 包含：
1. **Cloudflare Quick Tunnel 远程 SSH 穿透**：一键生成终端直连通道，让本地助手可直接操控云端 GPU。
2. **经典深度学习与金融量化实验**：PyTorch GPU 张量加速测试、多周期量价特征工程与 LightGBM 拟合。
3. **云端音视频处理套件测试**：FFmpeg 环境与 AI 语音转字幕。

## 步骤 1：启动 Cloudflare Tunnel 与 SSH 服务
点击运行下方代码，等待数秒即可看到生成的 SSH 直连命令。

In [ ]:
# 1. 安装配置 SSH
!apt-get update -qq && apt-get install -y openssh-server -qq
!mkdir -p /var/run/sshd
!echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
!echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
!echo "root:colab123" | chpasswd
!service ssh restart

# 2. 下载官方 cloudflared
!curl -sSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 3. 启动隧道并获取连接地址
import subprocess, time, re
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "tcp://localhost:22"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

hostname = None
start_time = time.time()
while time.time() - start_time < 20:
    line = proc.stdout.readline()
    if line:
        match = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)
        if match:
            hostname = match.group(1)
            break
    time.sleep(0.5)

if hostname:
    print("\n" + "=" * 60)
    print("🎉 Colab SSH 隧道已就绪！")
    print(f"穿透域名: {hostname}")
    print(f"SSH 连接命令: ssh -o ProxyCommand=\"cloudflared access ssh --hostname {hostname}\" root@{hostname}")
    print("登录密码: colab123")
    print("=" * 60)
else:
    print("未能自动捕获到域名，请查看日志。")

## 步骤 2：经典机器学习与金融多周期量价实验
测试 GPU 矩阵算力以及基于多周期形态（如均线翻转斜率与量能变化）的分类与夏普回测。

In [ ]:
import torch, time
import numpy as np

# 1. GPU 检测
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 显卡型号:", torch.cuda.get_device_name(0))
    print(f"GPU 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# 2. 深度学习神经网络梯度反向传播
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4096, 128, device=device)
y = torch.randn(4096, 1, device=device)
model = torch.nn.Sequential(
    torch.nn.Linear(128, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 1)
).to(device)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
for _ in range(100):
    loss = loss_fn(model(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"✅ 深度学习 100 Epochs 训练完成，耗时: {time.time()-t0:.4f} 秒")

# 3. 金融多周期量价形态与 GBDT 模拟
try:
    import lightgbm as lgb
    np.random.seed(42)
    n = 10000
    ma_slope = np.random.randn(n)        # 60分均线斜率
    vol_ratio = np.random.exponential(1.0, n)  # 突破放量倍数
    bias = np.random.randn(n)            # 15分乖离率
    y = ((0.12 * ma_slope + 0.08 * vol_ratio + np.random.normal(0, 0.5, n)) > 0).astype(int)
    X = np.column_stack([ma_slope, vol_ratio, bias])
    clf = lgb.LGBMClassifier(n_estimators=50, max_depth=3, verbose=-1)
    clf.fit(X[:8000], y[:8000])
    acc = np.mean(clf.predict(X[8000:]) == y[8000:])
    print(f"✅ 金融 LightGBM 因子重要性: {clf.feature_importances_} | 测试集准确率: {acc*100:.2f}%")
except Exception as e:
    print("LightGBM 运行跳过:", e)